In [1]:
import sys
import os
import pandas as pd

# Add the project root to the Python path
sys.path.append(os.path.join(os.getcwd(), '..'))

# Reload modules when code is changed (uncomment for development)
%load_ext autoreload
%autoreload 2

# 1. Project overview

What is the goal of this model?
* To see which factors have an influence on viewing figures.
* To be able to predict viewing figures.

How will the model be used?
* Program planning: when is the best time to broadcast a new program?
* Selling advertising blocks based on the number of viewers we have at certain times.

Input:
* Ratings data (top 20 Flemish programs from 2016-10-01)
* Weather, sunrise and sunset data (open-meteo data)

Which techniques do we use?
* Supervised learning: we have labels (ratings)
* Regression: we're predicting a number
* Batch learning: there is no continuous flow of new data

Performance measures:
* MAPE
* RMSE
* MAE

*We measure the distance between our prediction vector and the target values vector. In percentages for MAPE and in the number of viewers for RMSE and MAE*

# 2. Collect data

## 2.1 Ratings data

You can see how the data is collected and transformed in [src/ratings_data.py](./src/ratings_data.py).

In [2]:
ratings_df = pd.read_parquet('../data/ratings_data.parquet')

In [3]:
ratings_df.head()

,show,channel,date,start,duration,viewers
0,HET 7 UUR-JOURNAAL,EEN,2016-10-01,2016-10-01 19:00:05,0 days 00:31:38,721850
1,FC DE KAMPIOENEN,EEN,2016-10-01,2016-10-01 20:41:00,0 days 00:38:39,709606
2,WEG ZIJN WIJ,EEN,2016-10-01,2016-10-01 20:13:36,0 days 00:24:44,548239
3,IEDEREEN BEROEMD,EEN,2016-10-01,2016-10-01 19:38:10,0 days 00:29:01,523610
4,COMEDY TOPPERS,VTM,2016-10-01,2016-10-01 19:52:06,0 days 00:24:40,496216


In [4]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66048 entries, 0 to 66138
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype          
---  ------    --------------  -----          
 0   show      66048 non-null  object         
 1   channel   66048 non-null  object         
 2   date      66048 non-null  datetime64[ns] 
 3   start     66048 non-null  datetime64[ns] 
 4   duration  66048 non-null  timedelta64[ns]
 5   viewers   66048 non-null  int64          
dtypes: datetime64[ns](2), int64(1), object(2), timedelta64[ns](1)
memory usage: 3.5+ MB


## 2.2 Weather data

In [5]:
weather_df = pd.read_parquet('../data/weather_data.parquet')
weather_df.head()

,date,weather_code,temperature_2m_mean,temperature_2m_max,temperature_2m_min,sunrise,sunset,daylight_duration,sunshine_duration,precipitation_sum,rain_sum,snowfall_sum,precipitation_hours,wind_speed_10m_max,wind_gusts_10m_max,sunrise_time,sunset_time
0,2016-10-01,53.0,14.094833,18.836500,10.586500,1475300633,1475342431,41795.472656,36650.582031,1.2,1.2,0.0,3.0,18.391737,45.719997,2016-10-01 07:43:53+02:00,2016-10-01 19:20:31+02:00
1,2016-10-02,53.0,12.326084,14.836500,10.336500,1475387128,1475428700,41569.531250,34481.562500,1.3,1.3,0.0,4.0,25.582806,51.480000,2016-10-02 07:45:28+02:00,2016-10-02 19:18:20+02:00
2,2016-10-03,53.0,13.478168,17.636499,10.486501,1475473624,1475514969,41342.718750,33656.269531,0.7,0.7,0.0,2.0,16.516901,36.000000,2016-10-03 07:47:04+02:00,2016-10-03 19:16:09+02:00
3,2016-10-04,3.0,12.767751,16.086500,9.836500,1475560119,1475601237,41115.175781,16417.371094,0.0,0.0,0.0,0.0,21.288757,38.160000,2016-10-04 07:48:39+02:00,2016-10-04 19:13:57+02:00
4,2016-10-05,1.0,10.753169,14.086500,8.086500,1475646615,1475687505,40887.046875,37022.519531,0.0,0.0,0.0,0.0,24.640940,48.239998,2016-10-05 07:50:15+02:00,2016-10-05 19:11:45+02:00


In [6]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3295 entries, 0 to 3294
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype                          
---  ------               --------------  -----                          
 0   date                 3295 non-null   datetime64[ns]                 
 1   weather_code         3295 non-null   float32                        
 2   temperature_2m_mean  3295 non-null   float32                        
 3   temperature_2m_max   3295 non-null   float32                        
 4   temperature_2m_min   3295 non-null   float32                        
 5   sunrise              3295 non-null   int64                          
 6   sunset               3295 non-null   int64                          
 7   daylight_duration    3295 non-null   float32                        
 8   sunshine_duration    3295 non-null   float32                        
 9   precipitation_sum    3295 non-null   float32                        
 10  

## 2.3 Merge ratings and weather data

In [7]:
from src.transform.data_transformer import DataTransformer

data_transformer = DataTransformer()

In [8]:
data = data_transformer.merge_ratings_and_weather_data(ratings_df, weather_df)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype                          
---  ------               --------------  -----                          
 0   show                 66048 non-null  object                         
 1   channel              66048 non-null  object                         
 2   date                 66048 non-null  datetime64[ns]                 
 3   start                66048 non-null  datetime64[ns]                 
 4   duration             66048 non-null  timedelta64[ns]                
 5   viewers              66048 non-null  int64                          
 6   weather_code         66048 non-null  float32                        
 7   temperature_2m_mean  66048 non-null  float32                        
 8   temperature_2m_max   66048 non-null  float32                        
 9   temperature_2m_min   66048 non-null  float32                        
 10

## 2.4 Feature engineering

In [9]:
data = data_transformer.create_new_features(data)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 40 columns):
 #   Column                 Non-Null Count  Dtype                          
---  ------                 --------------  -----                          
 0   show                   66048 non-null  object                         
 1   channel                66048 non-null  object                         
 2   date                   66048 non-null  datetime64[ns]                 
 3   start                  66048 non-null  datetime64[ns]                 
 4   duration               66048 non-null  int64                          
 5   viewers                66048 non-null  int64                          
 6   weather_code           66048 non-null  float32                        
 7   temperature_2m_mean    66048 non-null  float32                        
 8   temperature_2m_max     66048 non-null  float32                        
 9   temperature_2m_min     66048 non-null  float32    

In [10]:
data = data_transformer.select_features(data)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   show                   66048 non-null  object 
 1   channel                66048 non-null  object 
 2   viewers                66048 non-null  int64  
 3   year                   66048 non-null  int32  
 4   month                  66048 non-null  int32  
 5   day_of_week            66048 non-null  int32  
 6   week                   66048 non-null  UInt32 
 7   covid_19               66048 non-null  bool   
 8   lockdown_1             66048 non-null  bool   
 9   lockdown_2             66048 non-null  bool   
 10  start_of_program_hour  66048 non-null  int32  
 11  end_of_program_hour    66048 non-null  int32  
 12  duration               66048 non-null  int64  
 13  in_primetime           66048 non-null  bool   
 14  ends_in_primetime      66048 non-null  bool   
 15  st

# 3. Data exploration

In [11]:
# Descriptive statistics of the label
data["viewers"].describe()

count    6.604800e+04
mean     4.403526e+05
std      2.751052e+05
min      1.588700e+04
25%      2.271048e+05
50%      3.552540e+05
75%      5.969795e+05
max      2.494114e+06
Name: viewers, dtype: float64

See v1/create_model.ipynb for data exploration

In [12]:
# Correlation with the label
data.corrwith(data["viewers"], numeric_only=True).sort_values(ascending=False)

viewers                  1.000000
ends_in_primetime        0.172677
end_of_program_hour      0.125076
start_of_program_hour    0.123127
lockdown_2               0.089155
covid_19                 0.077710
in_primetime             0.077645
wind_speed_10m_max       0.070513
sunrise_delta_end        0.068570
lockdown_1               0.057519
sunrise_delta_start      0.056922
wind_gusts_10m_max       0.056861
precipitation_hours      0.053751
weather_code             0.040045
snowfall_sum             0.030070
precipitation_sum        0.028010
rain_sum                 0.023948
starts_in_primetime      0.006538
week                    -0.033335
month                   -0.037321
duration                -0.092421
year                    -0.092958
day_of_week             -0.126386
sunshine_duration       -0.143190
temperature_2m_min      -0.171793
daylight_duration       -0.181748
temperature_2m_mean     -0.187319
temperature_2m_max      -0.189820
sunset_delta_end        -0.196418
sunset_delta_s

# 4. Model data preparation

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66048 entries, 0 to 66047
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   show                   66048 non-null  object 
 1   channel                66048 non-null  object 
 2   viewers                66048 non-null  int64  
 3   year                   66048 non-null  int32  
 4   month                  66048 non-null  int32  
 5   day_of_week            66048 non-null  int32  
 6   week                   66048 non-null  UInt32 
 7   covid_19               66048 non-null  bool   
 8   lockdown_1             66048 non-null  bool   
 9   lockdown_2             66048 non-null  bool   
 10  start_of_program_hour  66048 non-null  int32  
 11  end_of_program_hour    66048 non-null  int32  
 12  duration               66048 non-null  int64  
 13  in_primetime           66048 non-null  bool   
 14  ends_in_primetime      66048 non-null  bool   
 15  st

In [14]:
from sklearn.model_selection import train_test_split

# Split X and y
X = data.drop(columns=["viewers", "show"])
y = data["viewers"]

# Create training and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(52838, 31)
(13210, 31)
(52838,)
(13210,)


In [15]:
# Split categorical, binary and numerical features
categorical_ix = data_transformer.CATEGORICAL_FEATURES

binary_ix = data_transformer.BINARY_FEATURES

numerical_ix = X_train.columns.difference(categorical_ix + binary_ix)

print(numerical_ix)
print(categorical_ix)
print(binary_ix)

Index(['daylight_duration', 'duration', 'end_of_program_hour',
       'precipitation_hours', 'precipitation_sum', 'rain_sum', 'snowfall_sum',
       'start_of_program_hour', 'sunrise_delta_end', 'sunrise_delta_start',
       'sunset_delta_end', 'sunset_delta_start', 'sunshine_duration',
       'temperature_2m_max', 'temperature_2m_mean', 'temperature_2m_min',
       'wind_gusts_10m_max', 'wind_speed_10m_max'],
      dtype='object')
['year', 'month', 'day_of_week', 'week', 'channel', 'weather_code']
['covid_19', 'lockdown_1', 'lockdown_2', 'in_primetime', 'ends_in_primetime', 'starts_in_primetime', 'has_commercials']


In [16]:
# Create column transformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

col_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_ix),
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), categorical_ix),
        ("bin", "passthrough", binary_ix)
    ],
    remainder="drop" # drops features that are not (useful in production)
)

col_transformer

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 Index(['daylight_duration', 'duration', 'end_of_program_hour',
       'precipitation_hours', 'precipitation_sum', 'rain_sum', 'snowfall_sum',
       'start_of_program_hour', 'sunrise_delta_end', 'sunrise_delta_start',
       'sunset_delta_end', 'sunset_delta_start', 'sunshine_duration',
       'temperature_2m_max', 'temperature_2m_mea...e_2m_min',
       'wind_gusts_10m_max', 'wind_speed_10m_max'],
      dtype='object')),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['year', 'month', 'day_of_week', 'week',
                                  'channel', 'weather_code']),
                                ('bin', 'passthrough',
                                 ['covid_19', 'lockdown_1', 'lockdown_2',
                                  'in_primetime', 'ends_in_primetime',
                                  'starts_in_primetime', 'has_commercials'])])

# 5. Testing models

In [17]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

regressors = [
    ('lr', LinearRegression()),
    ('dtr', DecisionTreeRegressor(random_state=42)),
    ('lsvr', LinearSVR(epsilon=1.5, random_state=42)),
    ('svr', SVR(kernel="poly", degree=2, C=100, epsilon=0.1)),
    ('rfr', RandomForestRegressor(random_state=42)),
    ('gbr', GradientBoostingRegressor(random_state=42)),
    ('ridge', Ridge(alpha=1.0, solver="cholesky")),
    ('lasso', Lasso(alpha=0.1)),
    ('elasticnet', ElasticNet(alpha=0.1, l1_ratio=0.5)),
    ('abr', AdaBoostRegressor(random_state=42)), # Default DT regressor met max_depth=3
    ('xgb', XGBRegressor(random_state=42)),
    ('lgbm', LGBMRegressor(random_state=42))
]

for name, reg in regressors:
    print(name)
    pipeline = Pipeline(
        steps=[
            ('preprocessing', col_transformer),
            ('regressor', reg)
        ]
    )
    scores = -cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_percentage_error')
    print(f"MAPE: {scores.mean():.3f} (+/- {scores.std():.3f})")
    print("---")

lr


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


MAPE: 0.501 (+/- 0.005)
---
dtr


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


MAPE: 0.244 (+/- 0.002)
---
lsvr


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001576 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2957
[LightGBM] [Info] Number of data points in the train set: 42271, number of used features: 125
[LightGBM] [Info] Start training from score 439545.190864


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001472 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2955
[LightGBM] [Info] Number of data points in the train set: 42271, number of used features: 124
[LightGBM] [Info] Start training from score 439381.075844


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MAPE: 0.221 (+/- 0.003)
---


I can ignore the unknown category warning here. handle_unknown="ignore" is a good fallback to handle these edge cases. 

There is a performance impact but it's the same for all models.

This part of the notebook is just to compare the different models so the comparison is still valid.

Random forest haalt het beste resultaat: MAPE: 0.187 (+/- 0.003)

Kan het nog beter met een stackingregressor?

In [18]:
from sklearn.ensemble import StackingRegressor
from sklearn.metrics import mean_absolute_percentage_error

# Take best regressors as estimators
estimators = [
    ('dtr', DecisionTreeRegressor(random_state=42)),
    ('rf', RandomForestRegressor(random_state=42)),
    ('xgb', XGBRegressor(random_state=42))
]

# Try some final estimators
final_estimators = {
    'lgbm': LGBMRegressor(random_state=42),
    'rf': RandomForestRegressor(random_state=42)
}

# Compare different stacking configurations
for name, est in final_estimators.items():
    stacking_reg = StackingRegressor(
        estimators=estimators,
        final_estimator=est,
        cv=5
    )
    
    pipeline = Pipeline(
        steps=[
            ('preprocessing', col_transformer),
            ('regressor', stacking_reg)
        ]
    )
    
    scores = -cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_percentage_error')
    print(f"Stacking with {name} as final estimator:")
    print(f"MAPE: {scores.mean():.3f} (+/- {scores.std():.3f})")
    print("---")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000558 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 42270, number of used features: 3
[LightGBM] [Info] Start training from score 440170.844618


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000549 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 42270, number of used features: 3
[LightGBM] [Info] Start training from score 440077.301490


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000535 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 42270, number of used features: 3
[LightGBM] [Info] Start training from score 439825.314502


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000553 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 42271, number of used features: 3
[LightGBM] [Info] Start training from score 439381.075844


/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Stacking with lgbm as final estimator:
MAPE: 0.181 (+/- 0.003)
---


/opt/anaconda3/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Stacking with rf as final estimator:
MAPE: 0.192 (+/- 0.003)
---
